# AutismAssist AI — Therapy/Support Recommendation

Multi-label classification for five support areas in the provided dataset.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("../data/raw/asd_screening_dataset.csv")

features = [
    "Age","Gender","Eye_Contact","Response_to_Name",
    "Communication_Level","Joint_Attention","Sensory_Regulation",
    "Motor_Skills","ADL_Skills","Hyperactivity_Control",
    "Social_Interaction","ASD_Status"
]
targets = [
    "Speech_Therapy_Need","Occupational_Therapy_Need",
    "Behaviour_Therapy_Need","Parent_Training_Need",
    "Social_Skills_Support_Need"
]

X = df[features]
Y = df[targets]
cat = ["Gender"]
num = [c for c in features if c not in cat]
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)


In [ ]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])

model = Pipeline([
    ("preprocessor", pre),
    ("model", MultiOutputClassifier(
        RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")
    ))
])

model.fit(X_train, Y_train)
pred = model.predict(X_test)

for i, col in enumerate(targets):
    print(col, "accuracy=", round(accuracy_score(Y_test.iloc[:,i], pred[:,i]),4),
          "f1=", round(f1_score(Y_test.iloc[:,i], pred[:,i], zero_division=0),4))


In [ ]:
import joblib
joblib.dump(model, "../models/therapy_recommendation_model.joblib")


## Safety note

The model predicts labels present in the dataset. These outputs are not individualized clinical treatment recommendations and require review by qualified professionals.